In [16]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import re

# ============================================================
# Paste your SMODES output here (as-is)
# ============================================================
SMODES_TEXT = r"""
atom  type   position
  1   Mn     0.00000   0.00000   0.00000
  2   Mn     0.00000   0.00000  40.92600
  3   Bi     0.00000   0.00000  17.38946
  4   Bi     0.00000   0.00000  58.31546
  5   Bi     0.00000   0.00000  64.46254
  6   Bi     0.00000   0.00000  23.53654
  7   Te1    0.00000   0.00000   5.45953
  8   Te1    0.00000   0.00000  46.38553
  9   Te1    0.00000   0.00000  76.39247
 10   Te1    0.00000   0.00000  35.46647
 11   Te2    0.00000   0.00000  12.03634
 12   Te2    0.00000   0.00000  52.96234
 13   Te2    0.00000   0.00000  69.81566
 14   Te2    0.00000   0.00000  28.88966
Symmetry modes:
atom  type   displacement
------------------------------------------
  1   Mn     0.00000   0.00000   1.00000
  2   Mn    -0.00000  -0.00000  -1.00000
------------------------------------------
  3   Bi     0.00000   0.00000   1.00000
  4   Bi    -0.00000  -0.00000  -1.00000
  5   Bi     0.00000   0.00000   1.00000
  6   Bi    -0.00000  -0.00000  -1.00000
------------------------------------------
  7   Te1    0.00000   0.00000   1.00000
  8   Te1   -0.00000  -0.00000  -1.00000
  9   Te1    0.00000   0.00000   1.00000
 10   Te1   -0.00000  -0.00000  -1.00000
------------------------------------------
 11   Te2    0.00000   0.00000   1.00000
 12   Te2   -0.00000  -0.00000  -1.00000
 13   Te2    0.00000   0.00000   1.00000
 14   Te2   -0.00000  -0.00000  -1.00000
"""

# ============================================================
# Parsing utilities
# ============================================================

ATOM_LINE_RE = re.compile(
    r"^\s*(\d+)\s+([A-Za-z0-9_]+)\s+([-+0-9.]+)\s+([-+0-9.]+)\s+([-+0-9.]+)\s*$"
)

def parse_atoms_and_modes(smodes_text: str):
    """
    Returns:
      atom_ids: (N,) int (1-based)
      atom_types: (N,) str
      atom_positions: (N,3) float  (x,y,z)
      mode_blocks: list of blocks; each block is list of (atom_id, atom_type, disp_xyz)
    """
    lines = [ln.rstrip("\n") for ln in smodes_text.splitlines()]

    # --- parse atom positions until "Symmetry modes:"
    atom_ids = []
    atom_types = []
    atom_positions = []

    i = 0
    while i < len(lines) and "Symmetry modes:" not in lines[i]:
        m = ATOM_LINE_RE.match(lines[i])
        if m:
            atom_ids.append(int(m.group(1)))
            atom_types.append(m.group(2))
            atom_positions.append([float(m.group(3)), float(m.group(4)), float(m.group(5))])
        i += 1

    atom_ids = np.array(atom_ids, dtype=int)
    atom_types = np.array(atom_types, dtype=object)
    atom_positions = np.array(atom_positions, dtype=float)

    # --- parse symmetry mode blocks separated by dashed lines
    # Find where modes start:
    while i < len(lines) and "Symmetry modes:" not in lines[i]:
        i += 1
    if i == len(lines):
        raise ValueError("Could not find 'Symmetry modes:' in the SMODES text.")
    i += 1  # move past "Symmetry modes:"

    mode_blocks = []
    current_block = []

    # A dashed separator line signals either start/end of a block
    def flush_block():
        nonlocal current_block
        if current_block:
            mode_blocks.append(current_block)
            current_block = []

    while i < len(lines):
        ln = lines[i].strip()
        if ln.startswith("-"):
            # separator: end current block (if any)
            flush_block()
            i += 1
            continue

        m = ATOM_LINE_RE.match(lines[i])
        if m:
            atom_id = int(m.group(1))
            atype = m.group(2)
            disp = np.array([float(m.group(3)), float(m.group(4)), float(m.group(5))], dtype=float)
            current_block.append((atom_id, atype, disp))
        i += 1

    flush_block()
    return atom_ids, atom_types, atom_positions, mode_blocks


# ============================================================
# Plotting / animation
# ============================================================

def make_mode_gif(
    atom_ids,
    atom_types,
    atom_positions,
    mode_block,
    *,
    output_path,
    peak_displacement=0.7,   # in same units as positions (your z here looks like Bohr-ish, but keep consistent)
    frames_per_cycle=48,
    fps=12,
    dpi=170,
):
    """
    Makes a GIF for a single symmetry mode block.
    We animate along the z-axis only (because your mode is purely z).
    """

    # --- base coordinates
    z0 = atom_positions[:, 2].copy()

    # Sort for clean vertical stack
    sort_idx = np.argsort(z0)
    atom_ids_sorted = atom_ids[sort_idx]
    atom_types_sorted = atom_types[sort_idx]
    z0_sorted = z0[sort_idx]

    # --- build displacement array dz for all atoms (default 0)
    dz_full = np.zeros_like(z0, dtype=float)
    id_to_index = {aid: idx for idx, aid in enumerate(atom_ids)}  # in unsorted indexing

    for atom_id, _, disp_xyz in mode_block:
        dz_full[id_to_index[atom_id]] = disp_xyz[2]  # z component

    dz_sorted = dz_full[sort_idx]

    # Normalize so max |dz| = 1, so peak_displacement sets the visible amplitude
    max_abs = np.max(np.abs(dz_sorted))
    if max_abs > 0:
        dz_sorted = dz_sorted / max_abs

    # --- give each species a small x-offset so points don't overlap at x=0
    # Treat Te1 and Te2 as Te for styling, but keep label text in title if you want.
    def style_key(t):
        return "Te" if str(t).startswith("Te") else str(t)

    style_types = np.array([style_key(t) for t in atom_types_sorted], dtype=object)

    x0_sorted = np.zeros_like(z0_sorted)

    marker_sizes = {"Mn": 90, "Bi": 70, "Te": 55}
    marker_colors = {"Mn": "blue", "Bi": "#C2185B", "Te": "orange"}

    # --- Figure
    fig, ax = plt.subplots(figsize=(3.1, 7.2), dpi=dpi)
    ax.set_axis_off()

    # scatter per style species
    scatter_by_type = {}
    for t in ["Mn", "Bi", "Te"]:
        mask = (style_types == t)
        scatter_by_type[t] = ax.scatter(
            x0_sorted[mask], z0_sorted[mask],
            s=marker_sizes[t], c=marker_colors[t],
            linewidths=0
        )

    # legend
    legend_handles = [
        ax.scatter([], [], s=marker_sizes["Mn"], c=marker_colors["Mn"], linewidths=0),
        ax.scatter([], [], s=marker_sizes["Bi"], c=marker_colors["Bi"], linewidths=0),
        ax.scatter([], [], s=marker_sizes["Te"], c=marker_colors["Te"], linewidths=0),
    ]
    ax.legend(
        legend_handles, ["Mn", "Bi", "Te"],
        loc="upper right",
        frameon=False,
        handletextpad=0.4,
        borderpad=0.2,
        labelspacing=0.3
    )

    # limits
    ax.set_xlim(-0.55, 0.55)
    pad_y = 4.0
    ax.set_ylim(z0_sorted.min() - pad_y, z0_sorted.max() + pad_y)

    # Title describing which atoms move in this mode
    moving = np.where(np.abs(dz_sorted) > 1e-12)[0]
    moving_types = [atom_types_sorted[i] for i in moving]

    mode_terms = []
    for i, dz in enumerate(dz_sorted):
        if abs(dz) > 1e-12:
            atom_label = f"{atom_types_sorted[i]}({atom_ids_sorted[i]})"
            mode_terms.append(f"{atom_label}: {dz:+.1f}")

    # Wrap into multiple lines (e.g., 3 terms per line)
    terms_per_line = 3
    lines = [
        ", ".join(mode_terms[j:j+terms_per_line])
        for j in range(0, len(mode_terms), terms_per_line)
    ]
    title_text = "Mode vector:\n" + "\n".join(lines)
    # title = f"Symmetry mode: {', '.join(sorted(set(map(str, moving_types))))}"
    # fig.suptitle(title_text, x=0.02, ha="left", y=0.99, fontsize=8)
    # fig.subplots_adjust(top=0.88)

    def update(frame_idx):
        phase = 2 * np.pi * frame_idx / frames_per_cycle
        amp = peak_displacement * np.sin(phase)
        z = z0_sorted + amp * dz_sorted

        for t, sc in scatter_by_type.items():
            mask = (style_types == t)
            sc.set_offsets(np.column_stack([x0_sorted[mask], z[mask]]))

        return list(scatter_by_type.values())

    anim = FuncAnimation(fig, update, frames=frames_per_cycle, interval=1000 / fps, blit=True)
    
    anim.save(
        output_path,
        writer="pillow",
        fps=fps,
        savefig_kwargs={"bbox_inches": "tight", "pad_inches": 0.02},
    )
    plt.close(fig)


# ============================================================
# Main: parse + generate GIF per symmetry mode
# ============================================================

if __name__ == "__main__":
    atom_ids, atom_types, atom_positions, mode_blocks = parse_atoms_and_modes(SMODES_TEXT)

    print(f"Parsed {len(atom_ids)} atoms and {len(mode_blocks)} symmetry modes.")

    # Tweak these if you want:
    peak_displacement = 2.0   # IMPORTANT: your z positions are ~0..76; 0.7 may be too tiny visually
    frames_per_cycle = 48
    fps = 12

    for mode_index, mode_block in enumerate(mode_blocks, start=1):
        # Name output using which species participate
        species_in_mode = sorted(set(str(t) for _, t, _ in mode_block))
        species_tag = "_".join(species_in_mode)

        outname = f"smodes_mode_{mode_index:02d}_{species_tag}.gif"
        make_mode_gif(
            atom_ids,
            atom_types,
            atom_positions,
            mode_block,
            output_path=outname,
            peak_displacement=peak_displacement,
            frames_per_cycle=frames_per_cycle,
            fps=fps,
        )
        print("Saved:", outname)

Parsed 14 atoms and 4 symmetry modes.


Saved: smodes_mode_01_Mn.gif


Saved: smodes_mode_02_Bi.gif


Saved: smodes_mode_03_Te1.gif
Saved: smodes_mode_04_Te2.gif


In [19]:
EIGVECS = np.array([
    [-0.36369331, -0.43130046, -0.88762667,  0.69114685],
    [-0.54129087,  0.50409351,  0.12255009,  0.24709458],
    [-0.75810732, -0.50196940,  0.04129069, -0.43020842],
    [-0.00215743,  0.55488410, -0.44203557, -0.52552927],
], dtype=float)

FREQ_THz = np.array([8.118075, 1.043247, 6.305090, 4.894563], dtype=float)

# ============================================================
# Parsing utilities
# ============================================================
ATOM_LINE_RE = re.compile(
    r"^\s*(\d+)\s+([A-Za-z0-9_]+)\s+([-+0-9.]+)\s+([-+0-9.]+)\s+([-+0-9.]+)\s*$"
)

def build_symmetry_mode_displacements(atom_ids, mode_blocks):
    """
    Convert each SMODES block into a full displacement vector over all atoms.

    Returns:
      dz_sym: (n_sym, N) array of z-displacements (in the same units as given in SMODES)
    """
    N = len(atom_ids)
    id_to_index = {aid: idx for idx, aid in enumerate(atom_ids)}
    dz_sym = []

    for block in mode_blocks:
        dz = np.zeros(N, dtype=float)
        for atom_id, _, disp_xyz in block:
            dz[id_to_index[atom_id]] = disp_xyz[2]
        dz_sym.append(dz)

    return np.array(dz_sym, dtype=float)

def make_eigenmode_gif(
    atom_ids,
    atom_types,
    atom_positions,
    dz_eigen_full,
    *,
    freq_thz,
    mode_index,
    output_path,
    peak_displacement=2.0,
    fps=12,
    frames_base=48,
    f_ref=None,
    dpi=170,
):
    """
    Animate a phonon eigenmode described by dz_eigen_full (length N, in symmetry-mode basis units).

    Speed scaling:
      frames_per_cycle ~ frames_base * (f_ref / f_lambda)
    so higher f => fewer frames per cycle => visually faster oscillation.
    """
    z0 = atom_positions[:, 2].copy()

    # Sort for nice vertical stack
    sort_idx = np.argsort(z0)
    atom_ids_sorted = atom_ids[sort_idx]
    atom_types_sorted = atom_types[sort_idx]
    z0_sorted = z0[sort_idx]
    dz_sorted = dz_eigen_full[sort_idx].copy()

    # Normalize the eigenmode so max |dz| = 1, amplitude set by peak_displacement
    max_abs = np.max(np.abs(dz_sorted))
    if max_abs > 0:
        dz_sorted /= max_abs

    # Species styling
    def style_key(t):
        t = str(t)
        return "Te" if t.startswith("Te") else t

    style_types = np.array([style_key(t) for t in atom_types_sorted], dtype=object)
    x0_sorted = np.zeros_like(z0_sorted)

    marker_sizes = {"Mn": 90, "Bi": 70, "Te": 55}
    marker_colors = {"Mn": "blue", "Bi": "#C2185B", "Te": "orange"}

    fig, ax = plt.subplots(figsize=(3.1, 7.2), dpi=dpi)
    ax.set_axis_off()

    scatter_by_type = {}
    for t in ["Mn", "Bi", "Te"]:
        mask = (style_types == t)
        scatter_by_type[t] = ax.scatter(
            x0_sorted[mask], z0_sorted[mask],
            s=marker_sizes[t], c=marker_colors[t],
            linewidths=0
        )

    legend_handles = [
        ax.scatter([], [], s=marker_sizes["Mn"], c=marker_colors["Mn"], linewidths=0),
        ax.scatter([], [], s=marker_sizes["Bi"], c=marker_colors["Bi"], linewidths=0),
        ax.scatter([], [], s=marker_sizes["Te"], c=marker_colors["Te"], linewidths=0),
    ]
    ax.legend(
        legend_handles, ["Mn", "Bi", "Te"],
        loc="upper right",
        frameon=False,
        handletextpad=0.4,
        borderpad=0.2,
        labelspacing=0.3
    )

    ax.set_xlim(-0.55, 0.55)
    pad_y = 4.0
    ax.set_ylim(z0_sorted.min() - pad_y, z0_sorted.max() + pad_y)

    # Speed scaling
    if f_ref is None:
        f_ref = float(freq_thz)  # fallback
    frames_per_cycle = int(np.clip(np.round(frames_base * (f_ref / float(freq_thz))), 10, 96))

    # Title
    title = f"Eigenmode {mode_index}  |  f = {freq_thz:.3f} THz"
    # ax.text(0.02, 0.98, title, transform=ax.transAxes, va="top", ha="left", fontsize=9)

    def update(frame_idx):
        phase = 2 * np.pi * frame_idx / frames_per_cycle
        amp = peak_displacement * np.sin(phase)
        z = z0_sorted + amp * dz_sorted

        for t, sc in scatter_by_type.items():
            mask = (style_types == t)
            sc.set_offsets(np.column_stack([x0_sorted[mask], z[mask]]))

        return list(scatter_by_type.values())

    anim = FuncAnimation(fig, update, frames=frames_per_cycle, interval=1000 / fps, blit=True)
    anim.save(
        output_path,
        writer="pillow",
        fps=fps,
        savefig_kwargs={"bbox_inches": "tight", "pad_inches": 0.02},
    )
    plt.close(fig)

# ============================================================
# Main: parse + generate GIF per phonon eigenmode
# ============================================================
if __name__ == "__main__":
    atom_ids, atom_types, atom_positions, mode_blocks = parse_atoms_and_modes(SMODES_TEXT)

    print(f"Parsed {len(atom_ids)} atoms and {len(mode_blocks)} symmetry modes.")

    dz_sym = build_symmetry_mode_displacements(atom_ids, mode_blocks)  # (n_sym, N)

    n_sym, N = dz_sym.shape
    assert EIGVECS.shape[0] == n_sym, "EIGVECS first axis must match number of symmetry modes"

    # Use the lowest frequency as reference -> slowest oscillation
    f_ref = float(np.min(FREQ_THz))

    peak_displacement = 2.0
    fps = 12
    frames_base = 48

    for lam in range(EIGVECS.shape[1]):
        coeffs = EIGVECS[:, lam]  # (n_sym,)

        # Build eigenmode displacement over atoms: dz_eigen = sum_alpha e_{alpha,lam} dz_sym_alpha
        dz_eigen = np.tensordot(coeffs, dz_sym, axes=(0, 0))  # (N,)

        outname = f"phonon_eigenmode_{lam+1:02d}_f_{FREQ_THz[lam]:.3f}THz.gif"
        make_eigenmode_gif(
            atom_ids,
            atom_types,
            atom_positions,
            dz_eigen,
            freq_thz=FREQ_THz[lam],
            mode_index=lam + 1,
            output_path=outname,
            peak_displacement=peak_displacement,
            fps=fps,
            frames_base=frames_base,
            f_ref=f_ref,
        )
        print("Saved:", outname)

Parsed 14 atoms and 4 symmetry modes.


Saved: phonon_eigenmode_01_f_8.118THz.gif


Saved: phonon_eigenmode_02_f_1.043THz.gif


Saved: phonon_eigenmode_03_f_6.305THz.gif
Saved: phonon_eigenmode_04_f_4.895THz.gif
